In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, accuracy_score, classification_report


try:
    # Using the path provided by the user. Note: For Colab, you need to upload the file or mount Google Drive.
    df = pd.read_csv("/content/z.csv")
    print("Dataset loaded successfully.")
    print(df.head())

    # --- Data Cleaning and Preparation for z.csv ---
    # Convert 'rate' to numeric
    df['rate_numeric'] = df['rate'].apply(lambda x: float(x.split('/')[0]) if isinstance(x, str) and '/' in x and x != 'NEW' else np.nan)
    df['rate_numeric'].fillna(df['rate_numeric'].mean(), inplace=True)

    # Convert 'online_order' to numeric (0/1)
    df['online_order_numeric'] = df['online_order'].map({'Yes': 1, 'No': 0})
    df['online_order_numeric'].fillna(df['online_order_numeric'].mode()[0], inplace=True) # Fill NaNs with mode

    # Convert 'book_table' to numeric (0/1)
    df['book_table_numeric'] = df['book_table'].map({'Yes': 1, 'No': 0})
    df['book_table_numeric'].fillna(df['book_table_numeric'].mode()[0], inplace=True) # Fill NaNs with mode

    # Convert 'approx_cost(for two people)' to numeric
    df['approx_cost_numeric'] = df['approx_cost(for two people)'].astype(str).str.replace(',', '').astype(float)
    df['approx_cost_numeric'].fillna(df['approx_cost_numeric'].mean(), inplace=True)

    # 'votes' column
    df['votes'] = pd.to_numeric(df['votes'], errors='coerce')
    df['votes'].fillna(df['votes'].mean(), inplace=True)

    # Define features (X) and targets (y)
    X = df[['votes', 'rate_numeric', 'online_order_numeric']]
    y_linear = df['approx_cost_numeric']
    y_logistic = df['book_table_numeric'] # Predicting if booking a table is possible

except FileNotFoundError:
    print("Error: The specified dataset file (z.csv) was not found at the provided local path.")
    print("Please ensure 'z.csv' is uploaded to your Colab environment or provide the correct path (e.g., if it's in Google Drive). You can also upload it using the file browser on the left.")
    # Creating a dummy dataset for demonstration if file not found
    print("Creating a dummy dataset for demonstration purposes.")
    data = {
        'feature1': np.random.rand(100) * 10,
        'feature2': np.random.randint(1, 5, 100),
        'target_linear': np.random.rand(100) * 5 + np.random.rand(100) * 2,
        'target_logistic': np.random.randint(0, 2, 100)
    }
    df = pd.DataFrame(data)
    df['target_linear'] = df['feature1'] * 0.5 + df['feature2'] * 1.2 + np.random.randn(100) * 0.5
    df['target_logistic'] = (df['feature1'] + df['feature2'] > 7).astype(int)
    print(df.head())

    # Use dummy data for X and y if file not found
    X = df[['feature1', 'feature2']]
    y_linear = df['target_linear']
    y_logistic = df['target_logistic']

# --- 3. Linear Regression ---
print("\n--- Performing Linear Regression ---")

X_train_lr, X_test_lr, y_train_lr, y_test_lr = train_test_split(X, y_linear, test_size=0.2, random_state=42)

linear_model = LinearRegression()
linear_model.fit(X_train_lr, y_train_lr)

y_pred_lr = linear_model.predict(X_test_lr)

mse = mean_squared_error(y_test_lr, y_pred_lr)
rmse = np.sqrt(mse)
print(f"Mean Squared Error (Linear Regression): {mse:.2f}")
print(f"Root Mean Squared Error (Linear Regression): {rmse:.2f}")
print(f"Linear Regression Coefficients: {linear_model.coef_}")
print(f"Linear Regression Intercept: {linear_model.intercept_:.2f}")

# --- 4. Logistic Regression ---
print("\n--- Performing Logistic Regression ---")


X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(X, y_logistic, test_size=0.2, random_state=42)


logistic_model = LogisticRegression(random_state=42, solver='liblinear') # 'liblinear' is good for small datasets
logistic_model.fit(X_train_log, y_train_log)

y_pred_log = logistic_model.predict(X_test_log)
y_pred_proba_log = logistic_model.predict_proba(X_test_log)[:, 1] # Probability of the positive class

accuracy = accuracy_score(y_test_log, y_pred_log)
print(f"Accuracy (Logistic Regression): {accuracy:.2f}")
print("\nClassification Report (Logistic Regression):\n", classification_report(y_test_log, y_pred_log))
print(f"Logistic Regression Coefficients: {logistic_model.coef_}")
print(f"Logistic Regression Intercept: {logistic_model.intercept_[0]:.2f}")

Dataset loaded successfully.
                    name online_order book_table   rate  votes  \
0                  Jalsa          Yes        Yes  4.1/5    775   
1         Spice Elephant          Yes         No  4.1/5    787   
2        San Churro Cafe          Yes         No  3.8/5    918   
3  Addhuri Udupi Bhojana           No         No  3.7/5     88   
4          Grand Village           No         No  3.8/5    166   

   approx_cost(for two people) listed_in(type)  
0                          800          Buffet  
1                          800          Buffet  
2                          800          Buffet  
3                          300          Buffet  
4                          600          Buffet  

--- Performing Linear Regression ---
Mean Squared Error (Linear Regression): 44579.68
Root Mean Squared Error (Linear Regression): 211.14
Linear Regression Coefficients: [5.67349708e-02 3.50605042e+01 1.37035931e+02]
Linear Regression Intercept: 233.98

--- Performing Logistic R

/tmp/ipykernel_15171/2755064417.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['rate_numeric'].fillna(df['rate_numeric'].mean(), inplace=True)
/tmp/ipykernel_15171/2755064417.py:21: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value

### Decision Tree Classifier

In [ ]:
from sklearn.tree import DecisionTreeClassifier

print("\n--- Performing Decision Tree Classification ---")

decision_tree_model = DecisionTreeClassifier(random_state=42)
decision_tree_model.fit(X_train_log, y_train_log)

y_pred_dt = decision_tree_model.predict(X_test_log)

accuracy_dt = accuracy_score(y_test_log, y_pred_dt)
print(f"Accuracy (Decision Tree): {accuracy_dt:.2f}")
print("\nClassification Report (Decision Tree):\n", classification_report(y_test_log, y_pred_dt))


--- Performing Decision Tree Classification ---
Accuracy (Decision Tree): 0.93

Classification Report (Decision Tree):
               precision    recall  f1-score   support

           0       0.96      0.96      0.96        28
           1       0.50      0.50      0.50         2

    accuracy                           0.93        30
   macro avg       0.73      0.73      0.73        30
weighted avg       0.93      0.93      0.93        30



### Random Forest Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier

print("\n--- Performing Random Forest Classification ---")

random_forest_model = RandomForestClassifier(random_state=42)
random_forest_model.fit(X_train_log, y_train_log)

y_pred_rf = random_forest_model.predict(X_test_log)

accuracy_rf = accuracy_score(y_test_log, y_pred_rf)
print(f"Accuracy (Random Forest): {accuracy_rf:.2f}")
print("\nClassification Report (Random Forest):\n", classification_report(y_test_log, y_pred_rf))


--- Performing Random Forest Classification ---
Accuracy (Random Forest): 0.93

Classification Report (Random Forest):
               precision    recall  f1-score   support

           0       0.93      1.00      0.97        28
           1       0.00      0.00      0.00         2

    accuracy                           0.93        30
   macro avg       0.47      0.50      0.48        30
weighted avg       0.87      0.93      0.90        30



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### Support Vector Machine (SVM) Classifier

In [ ]:
from sklearn.svm import SVC

print("\n--- Performing SVM Classification ---")

# Note: SVM can be sensitive to feature scaling. For simplicity, we'll use it directly.
# For better performance, consider scaling features (e.g., StandardScaler) before applying SVM.
svm_model = SVC(random_state=42)
svm_model.fit(X_train_log, y_train_log)

y_pred_svm = svm_model.predict(X_test_log)

accuracy_svm = accuracy_score(y_test_log, y_pred_svm)
print(f"Accuracy (SVM): {accuracy_svm:.2f}")
print("\nClassification Report (SVM):\n", classification_report(y_test_log, y_pred_svm))


--- Performing SVM Classification ---
Accuracy (SVM): 0.93

Classification Report (SVM):
               precision    recall  f1-score   support

           0       0.93      1.00      0.97        28
           1       0.00      0.00      0.00         2

    accuracy                           0.93        30
   macro avg       0.47      0.50      0.48        30
weighted avg       0.87      0.93      0.90        30



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### Principal Component Analysis (PCA)

PCA is a dimensionality reduction technique. It's often used to reduce the number of features in your dataset while retaining as much variance as possible. This can help with model performance and visualization.

Since the current dataset has only 3 features, PCA might not offer significant dimensionality reduction benefit for *this specific feature set*, but it's a valuable technique to understand. I'll demonstrate it here, and you can apply it in scenarios with many more features.

We will apply PCA and then use the transformed data with a simple classifier (e.g., Decision Tree) to show its usage.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

print("\n--- Performing PCA ---")

# It's good practice to scale your data before applying PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply PCA to reduce dimensions
# Let's reduce it to 2 components for demonstration purposes
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"Original number of features: {X.shape[1]}")
print(f"Reduced number of features (PCA): {X_pca.shape[1]}")
print(f"Explained variance ratio by principal components: {pca.explained_variance_ratio_}")
print(f"Total explained variance: {pca.explained_variance_ratio_.sum():.2f}")

# Optionally, train a classifier on the PCA-transformed data
# We'll use logistic regression as an example
print("\n--- Logistic Regression after PCA ---")

X_train_pca, X_test_pca, y_train_pca, y_test_pca = train_test_split(X_pca, y_logistic, test_size=0.2, random_state=42)

logistic_model_pca = LogisticRegression(random_state=42, solver='liblinear')
logistic_model_pca.fit(X_train_pca, y_train_pca)

y_pred_log_pca = logistic_model_pca.predict(X_test_pca)

accuracy_log_pca = accuracy_score(y_test_pca, y_pred_log_pca)
print(f"Accuracy (Logistic Regression with PCA): {accuracy_log_pca:.2f}")
print("\nClassification Report (Logistic Regression with PCA):\n", classification_report(y_test_pca, y_pred_log_pca))


--- Performing PCA ---
Original number of features: 3
Reduced number of features (PCA): 2
Explained variance ratio by principal components: [0.62379629 0.21345599]
Total explained variance: 0.84

--- Logistic Regression after PCA ---
Accuracy (Logistic Regression with PCA): 0.93

Classification Report (Logistic Regression with PCA):
               precision    recall  f1-score   support

           0       0.93      1.00      0.97        28
           1       0.00      0.00      0.00         2

    accuracy                           0.93        30
   macro avg       0.47      0.50      0.48        30
weighted avg       0.87      0.93      0.90        30



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### Long Short-Term Memory (LSTM) Networks

LSTM networks are a type of recurrent neural network (RNN) primarily designed for sequence prediction problems, such as time series forecasting, natural language processing, or speech recognition. They are particularly good at learning long-term dependencies.

Your current dataset `z.csv` appears to be tabular data without an explicit sequential or temporal component that would naturally lend itself to an LSTM model. To use an LSTM, we would typically need to:

1.  **Structure the data as sequences:** This might involve creating sequences from time-series data (e.g., daily sales over several days as input to predict the next day's sales) or transforming other data into sequential form.
2.  **Reshape the input:** LSTM layers expect input data to be in a 3D format: `(samples, timesteps, features)`.

Given the nature of the `z.csv` data (static restaurant information), applying an LSTM directly would require significant feature engineering to create meaningful sequences. If you have a specific sequential aspect of the data in mind (e.g., tracking a restaurant's performance over time), please let me know, and we can explore how to prepare the data for an LSTM model.